<a href="https://colab.research.google.com/github/velascocafe23/nlp-maestria-upb/blob/main/04-cpt-smollm2/lab_cpt_smollm2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Entrega 4 · Lab CPT (SmolLM2)** — Sebastián Velasco Ardila · Maestría en Ciencia de Datos, UPB · Curso de NLP

Parte A: ejecutada tal como viene. Parte B: resuelta (pasos 11 a 15, más dos extras: olvido cruzado y tabla resumen).

# Continued Pretraining: Enseñándole a un Modelo un Dominio Nuevo

En este laboratorio vas a hacer **Continued Pretraining (CPT)**: tomar un modelo de lenguaje ya preentrenado y **seguir** entrenándolo con la misma tarea de preentrenamiento (predecir el siguiente token) pero sobre un **corpus de un dominio nuevo**.

La idea clave: el CPT **no** le enseña una tarea nueva al modelo (eso es fine-tuning supervisado). Le enseña un **estilo, vocabulario y distribución de texto** nuevos. El modelo sigue siendo un generador de texto, pero ahora "habla" más como el dominio objetivo.

**Estructura del lab:**
- **Parte A (guiada):** CPT completo de `SmolLM2-135M` con un corpus de **tweets en español**. Todo el código está resuelto.
- **Parte B (tu reto):** repetir el patrón con un corpus **financiero en español**. Tú completas el código con la ayuda de los hints.

> **Entorno:** Google Colab con GPU **T4**. Antes de empezar ve a `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`.


---
## Parte A: Ejemplo Guiado — CPT con tweets

Vamos a enseñarle al modelo a escribir como tweets en español.


### 1. Configuración del entorno

Instalamos las librerías, verificamos que tenemos la GPU T4 y definimos el dispositivo (`DEVICE`) que usaremos en todo el notebook.


In [ ]:
# Instalación de dependencias
!pip install -q transformers datasets torch

import torch

# Verificar que tenemos GPU (idealmente una T4)
print("¿CUDA disponible?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU detectada:", torch.cuda.get_device_name(0))
else:
    print("ADVERTENCIA: no hay GPU. Ve a 'Entorno de ejecución -> Cambiar tipo' y elige T4 GPU.")

# Dispositivo global
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)

# Semilla para reproducibilidad
torch.manual_seed(42)


### 2. Cargar el modelo base

Cargamos `SmolLM2-135M`, un modelo **decoder** (arquitectura tipo GPT/Llama) de ~135 millones de parámetros. Es pequeño a propósito: cabe cómodo en una T4 y podemos hacer *full fine-tuning* (entrenar todos sus pesos) sin trucos.

Usamos `AutoModelForCausalLM` porque es un modelo de **lenguaje causal**: predice el siguiente token dados los anteriores.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

# Los modelos causales suelen no traer pad_token; usamos el eos como padding.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Confirmamos que es un decoder de ~135M parámetros
n_params = sum(p.numel() for p in model.parameters())
print(f"Parámetros del modelo: {n_params:,} (~{n_params/1e6:.0f}M)")
print("Tipo de arquitectura:", model.config.model_type)


### 3. ANTES del CPT: ¿cómo escribe el modelo hoy?

Antes de tocar nada, veamos qué genera el modelo con prompts de nuestro dominio objetivo (tweets sobre servicios financieros en Colombia). Definimos una función de generación reutilizable.

Fíjate en el resultado: el modelo produce texto en español, pero **genérico** — no suena a tweet, no tiene el estilo corto, informal y con jerga de redes sociales.


In [ ]:
def generar(prompt, max_new_tokens=40):
    """Genera texto a partir de un prompt usando el modelo actual."""
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        salida = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(salida[0], skip_special_tokens=True)


PROMPTS_TWEETS = [
    "Los usuarios de Nequi reportan que",
    "El servicio de Bancolombia hoy está",
]

print("===== GENERACIÓN ANTES DEL CPT =====\n")
for p in PROMPTS_TWEETS:
    print(f"[Prompt] {p}")
    print(f"[Modelo] {generar(p)}\n")


### 4. Preparar el corpus de tweets

Cargamos el corpus de tweets en español desde sus archivos **JSONL**.

> Nota técnica: `datasets` 5.x eliminó el soporte para datasets basados en scripts de Python, así que cargamos los archivos JSONL directamente desde el repositorio. Los campos son `text` y `label`, y `label` viene como **string** (`"0"`, `"1"`, `"2"`), por lo que lo convertimos a `int`.

Punto clave del CPT: **ignoramos las etiquetas** (`label`). No nos importa el sentimiento. Solo queremos el **texto crudo** para que el modelo aprenda a escribir como un tweet.

El flujo estándar de CPT para un decoder:
1. Juntar todo el texto en una sola secuencia larga.
2. Tokenizar esa secuencia.
3. Partirla en **chunks** de longitud fija (aquí, 128 tokens). Cada chunk es un ejemplo de entrenamiento.


In [ ]:
from datasets import load_dataset
from torch.utils.data import TensorDataset, DataLoader

CHUNK_SIZE = 128
BATCH_SIZE = 16

# datasets 5.x no soporta scripts Python, cargamos directo desde JSONL
data_files = {
    "train": "https://huggingface.co/datasets/cardiffnlp/tweet_sentiment_multilingual/resolve/main/data/spanish/train.jsonl",
    "validation": "https://huggingface.co/datasets/cardiffnlp/tweet_sentiment_multilingual/resolve/main/data/spanish/validation.jsonl",
    "test": "https://huggingface.co/datasets/cardiffnlp/tweet_sentiment_multilingual/resolve/main/data/spanish/test.jsonl",
}
tweets = load_dataset("json", data_files=data_files)
# label viene como STRING ("0","1","2"); convertir a int
tweets = tweets.map(lambda x: {"label": int(x["label"])})
tweets_train = tweets["train"]
tweets_test = tweets["test"]

print("Ejemplos de tweets (train):", len(tweets_train))
print("Ejemplo de texto:", tweets_train[0]["text"])


def tokenize_corpus(textos, chunk_size=CHUNK_SIZE):
    """Concatena una lista de textos y la parte en chunks de tamaño fijo.

    Devuelve un tensor (n_chunks, chunk_size) con los input_ids.
    """
    # Unimos todos los textos en un solo string, separados por el token de fin de secuencia
    sep = tokenizer.eos_token or " "
    texto_unido = sep.join(t for t in textos if t and t.strip())

    # Tokenizamos TODO de una vez (sin truncar)
    ids = tokenizer(texto_unido, return_tensors="pt", truncation=False)["input_ids"][0]

    # Recortamos para que la longitud sea múltiplo exacto de chunk_size
    n_chunks = ids.size(0) // chunk_size
    ids = ids[: n_chunks * chunk_size]

    # Reshape a (n_chunks, chunk_size)
    chunks = ids.view(n_chunks, chunk_size)
    return chunks


# Tokenizar el corpus de tweets -> chunks
chunks_train = tokenize_corpus(tweets_train["text"])
print("Chunks de entrenamiento:", tuple(chunks_train.shape))

# DataLoader básico de PyTorch
train_ds = TensorDataset(chunks_train)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
print("Batches por epoch:", len(train_loader))


### 5. Medir perplexity ANTES del CPT

La **perplexity** mide qué tan "sorprendido" está el modelo por un texto. Perplexity baja = el modelo predice bien ese texto = el texto le resulta familiar.

La calculamos sobre un **holdout** (el split `test`, que el modelo no verá durante el entrenamiento). Guardamos este número para compararlo al final: esperamos que **baje** tras el CPT.


In [ ]:
import math

# Corpus de evaluación (holdout): split test ya cargado arriba, también solo texto
chunks_eval = tokenize_corpus(tweets_test["text"])
eval_ds = TensorDataset(chunks_eval)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False)


def calcular_perplexity(modelo, loader):
    """Perplexity promedio del modelo sobre un DataLoader de chunks."""
    modelo.eval()
    total_loss = 0.0
    total_batches = 0
    with torch.no_grad():
        for (batch,) in loader:
            batch = batch.to(DEVICE)
            out = modelo(input_ids=batch, labels=batch)
            total_loss += out.loss.item()
            total_batches += 1
    loss_promedio = total_loss / total_batches
    return math.exp(loss_promedio)


ppl_antes = calcular_perplexity(model, eval_loader)
print(f"Perplexity ANTES del CPT: {ppl_antes:.2f}")


### 6. Training loop (CPT explícito en PyTorch)

Aquí está el corazón del CPT. En vez de usar el `Trainer` de HuggingFace, escribimos el bucle **a mano** para que veas exactamente qué pasa:

- Para cada `batch`, le pasamos `input_ids` **y** `labels=input_ids`. El modelo desplaza internamente las etiquetas y calcula la pérdida de **predecir el siguiente token**.
- `loss.backward()` calcula gradientes; `optimizer.step()` actualiza **todos** los pesos (full fine-tuning).
- Repetimos por 3 epochs.

Deberías ver la `loss` **bajando** con los steps: el modelo se está adaptando al estilo de los tweets.


In [ ]:
from torch.optim import Adam

EPOCHS = 3
LR = 5e-5

optimizer = Adam(model.parameters(), lr=LR)

model.train()
global_step = 0
for epoch in range(EPOCHS):
    for (batch,) in train_loader:
        batch = batch.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=batch, labels=batch)  # LM causal: labels = input_ids
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        if global_step % 50 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} | step {global_step} | loss = {loss.item():.4f}")
        global_step += 1

    print(f"--- Fin epoch {epoch+1} | última loss = {loss.item():.4f} ---")

print("\nCPT terminado.")


### 7. DESPUÉS del CPT: ¿cambió el estilo?

Volvemos a generar con **los mismos prompts** del paso 3. Ahora el modelo debería producir texto que suena más a **tweet en español**: más informal, más corto, con el "sabor" de redes sociales del corpus.


In [ ]:
print("===== GENERACIÓN DESPUÉS DEL CPT =====\n")
for p in PROMPTS_TWEETS:
    print(f"[Prompt] {p}")
    print(f"[Modelo] {generar(p)}\n")


### 8. Medir perplexity DESPUÉS del CPT

Recalculamos la perplexity sobre el **mismo holdout** del paso 5 y comparamos. Un descenso confirma que el modelo ahora modela mejor el dominio de los tweets.


In [ ]:
ppl_despues = calcular_perplexity(model, eval_loader)

print(f"Perplexity ANTES  del CPT: {ppl_antes:.2f}")
print(f"Perplexity DESPUÉS del CPT: {ppl_despues:.2f}")
mejora = 100 * (ppl_antes - ppl_despues) / ppl_antes
print(f"Cambio relativo: {mejora:+.1f}%  (negativo = mejoró)")


### 9. Reflexión

Responde mentalmente (o discútelo con tu grupo) antes de seguir:

- **¿Qué aprendió el modelo?** Aprendió la *distribución* del texto del dominio (estilo, vocabulario, longitud, jerga), no una tarea nueva. Sigue siendo un generador de texto.
- **¿Cambió su conocimiento general?** El CPT sobre un corpus pequeño y estrecho puede provocar *olvido catastrófico*: el modelo mejora en el dominio nuevo pero puede empeorar en textos generales. La perplexity del dominio baja, pero la de otros dominios podría subir.
- **¿La perplexity bajó tanto como esperabas?** Con 3 epochs y un corpus pequeño el efecto es visible pero moderado. Más datos y más pasos = adaptación más fuerte (y más riesgo de olvido).

```
# === PAUSA: DISCUSIÓN ===
```
Antes de pasar a la Parte B, comenta con tus compañeros: si quisieras adaptar el modelo a tweets **sin** que olvide el español general, ¿qué harías? (pista: mezclar corpus, menos epochs, learning rate más bajo, LoRA...).


---
### Limpieza de memoria antes de la Parte B

Liberamos memoria de la GPU para que la Parte B tenga espacio.


In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("Caché de CUDA liberada.")
if torch.cuda.is_available():
    print(f"Memoria GPU reservada: {torch.cuda.memory_reserved(0)/1e9:.2f} GB")


---
## Parte B: Tu turno — CPT con corpus financiero

Ahora te toca a ti. El objetivo es el mismo (CPT), pero cambiando el **dominio**: en vez de tweets, un corpus de **frases financieras en español**.

**El patrón es idéntico al de la Parte A.** Solo cambian los datos. Puedes (y debes) **reutilizar las funciones** que ya definimos: `tokenize_corpus`, `calcular_perplexity`, `generar`.

> **Importante (parte del reto):** para medir el efecto del dominio financiero de forma limpia, **recarga el modelo base original** antes de entrenar. Así partes del mismo punto que en la Parte A y la comparación es justa (no arrastras la adaptación a tweets).


### 10. Recargar el modelo base (obligatorio)

Vuelve a cargar `SmolLM2-135M` desde cero para empezar el reto sin la adaptación a tweets.


In [ ]:
# Recargamos el modelo base para partir desde cero (sin la adaptación a tweets de la Parte A)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Modelo base recargado. Listo para el CPT financiero.")


### 11. Cargar el corpus financiero

Completa la celda para cargar y filtrar el dataset. Lee bien los hints: este dataset **no** tiene subset `spanish`.


In [ ]:
# Paso 11 (resuelto): cargar el corpus financiero y quedarnos con el español
REPO_FIN = "nojedag/financial_phrasebank_multilingual"

dataset_fin = load_dataset(REPO_FIN, split="train")   # un solo subset: "default"
print(dataset_fin)
print("\nColumnas y tipos:", dataset_fin.features)

# OJO con el hint: "lang" NO es un string. Es un ClassLabel, es decir un ENTERO
# con nombres ['en', 'fr', 'de', 'es'] (español = 3). Comparar x["lang"] == "es"
# compara un entero contra un string: no da error, pero devuelve 0 filas.
# En vez de dejar el 3 "quemado", le preguntamos al dataset cómo codifica "es".
def id_de_idioma(ds, codigo="es"):
    """Valor con el que el dataset codifica un idioma (entero si es ClassLabel)."""
    feat = ds.features["lang"]
    return feat.str2int(codigo) if hasattr(feat, "str2int") else codigo


ES = id_de_idioma(dataset_fin)
print(f'\nEl idioma "es" está codificado como: {ES!r}')

# Filtra solo las filas en español
dataset_fin_es = dataset_fin.filter(lambda x: x["lang"] == ES)
assert len(dataset_fin_es) > 0, "El filtro dejó 0 filas: revisa cómo viene codificado 'lang'."

print("Frases financieras en español:", len(dataset_fin_es))
print("Ejemplo:", dataset_fin_es[0]["sentence"])

### 12. Tokenizar y crear el DataLoader

Reutiliza `tokenize_corpus`. El único cambio real frente a la Parte A: el campo se llama `"sentence"`, no `"text"`.


In [ ]:
# Paso 12 (resuelto): tokenizar el corpus financiero y crear el DataLoader
# Igual que en el paso 4, pero el texto crudo está en "sentence" (antes era "text").
# Las etiquetas de sentimiento ("labels") se ignoran: en CPT solo importa el texto.
textos_fin = list(dataset_fin_es["sentence"])

chunks_fin = tokenize_corpus(textos_fin)                 # (n_chunks, CHUNK_SIZE)
train_ds_fin = TensorDataset(chunks_fin)
train_loader_fin = DataLoader(train_ds_fin, batch_size=BATCH_SIZE, shuffle=True)

print("Chunks financieros:", tuple(chunks_fin.shape))
print("Batches por epoch:", len(train_loader_fin))
print(f"(Referencia Parte A: tweets = {tuple(chunks_train.shape)} chunks, {len(train_loader)} batches)")

### 13. Medir perplexity ANTES (dominio financiero)

Aplica la función `calcular_perplexity` que ya existe. Necesitas un holdout financiero: usa el split `test` del dataset, filtrado a español.


In [ ]:
# Paso 13 (resuelto): perplexity del modelo BASE sobre el holdout financiero
fin_test = load_dataset(REPO_FIN, split="test")
fin_test_es = fin_test.filter(lambda x: x["lang"] == ES)
assert len(fin_test_es) > 0, "El holdout financiero en español quedó vacío."

chunks_eval_fin = tokenize_corpus(list(fin_test_es["sentence"]))
eval_loader_fin = DataLoader(TensorDataset(chunks_eval_fin), batch_size=BATCH_SIZE, shuffle=False)
print("Frases de holdout:", len(fin_test_es), "| chunks:", tuple(chunks_eval_fin.shape))

ppl_fin_antes = calcular_perplexity(model, eval_loader_fin)
print(f"Perplexity financiera ANTES del CPT: {ppl_fin_antes:.2f}")

# Extra: guardamos cómo escribe el modelo base con prompts financieros, para
# poder comparar lado a lado en el paso 15 (el enunciado solo pide el "después").
PROMPTS_FINANCIEROS = [
    "El Banco de la República decidió",
    "Las acciones de Ecopetrol",
    "La inflación en Colombia",
]
torch.manual_seed(42)   # misma semilla antes y después: cambia el modelo, no el azar
gen_fin_antes = {p: generar(p) for p in PROMPTS_FINANCIEROS}

print("\n===== GENERACIÓN ANTES DEL CPT FINANCIERO =====\n")
for p, texto in gen_fin_antes.items():
    print(f"[Prompt] {p}")
    print(f"[Modelo] {texto}\n")

### 14. Training loop del CPT financiero

Misma estructura que el paso 6. Tú decides los hiperparámetros. Empieza con valores parecidos a la Parte A (`epochs=3`, `lr=5e-5`) y experimenta.


In [ ]:
# Paso 14 (resuelto): training loop de CPT sobre el corpus financiero
# Usamos los MISMOS hiperparámetros de la Parte A (3 epochs, lr 5e-5). Así, si la
# perplexity baja distinto que con los tweets, la diferencia se debe al dominio
# y al tamaño del corpus, no a una configuración de entrenamiento diferente.
EPOCHS_FIN = 3
LR_FIN = 5e-5

optimizer_fin = Adam(model.parameters(), lr=LR_FIN)   # optimizador nuevo para el modelo recargado
historial_loss_fin = []

model.train()
global_step = 0
for epoch in range(EPOCHS_FIN):
    suma_loss, n_batches = 0.0, 0
    for (batch,) in train_loader_fin:
        batch = batch.to(DEVICE)

        optimizer_fin.zero_grad()
        outputs = model(input_ids=batch, labels=batch)   # LM causal: labels = input_ids
        loss = outputs.loss

        loss.backward()
        optimizer_fin.step()

        historial_loss_fin.append(loss.item())
        suma_loss += loss.item()
        n_batches += 1

        if global_step % 50 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS_FIN} | step {global_step} | loss = {loss.item():.4f}")
        global_step += 1

    print(f"--- Fin epoch {epoch+1} | loss promedio del epoch = {suma_loss / n_batches:.4f} ---")

print("\nCPT financiero terminado.")

# Curva de pérdida: debería bajar de forma sostenida.
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 3))
plt.plot(historial_loss_fin, linewidth=1)
plt.xlabel("step")
plt.ylabel("loss (siguiente token)")
plt.title("CPT financiero: pérdida de entrenamiento")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 15. Perplexity DESPUÉS + generación financiera

Recalcula la perplexity financiera y genera con prompts del dominio. Reutiliza `generar` y `calcular_perplexity`.


In [ ]:
# Paso 15 (resuelto)
# (a) Perplexity financiera DESPUÉS del CPT, sobre el mismo holdout del paso 13
ppl_fin_despues = calcular_perplexity(model, eval_loader_fin)
print(f"Perplexity financiera ANTES:   {ppl_fin_antes:.2f}")
print(f"Perplexity financiera DESPUÉS: {ppl_fin_despues:.2f}")
reduccion_fin = 100 * (ppl_fin_antes - ppl_fin_despues) / ppl_fin_antes
print(f"Reducción relativa: {reduccion_fin:.1f}%  (positivo = el modelo predice mejor el dominio)")

# (b) Generación con prompts financieros (definidos en el paso 13), lado a lado
torch.manual_seed(42)   # misma semilla que en el "antes"
gen_fin_despues = {p: generar(p) for p in PROMPTS_FINANCIEROS}

print("\n===== GENERACIÓN FINANCIERA: ANTES vs. DESPUÉS DEL CPT =====\n")
for p in PROMPTS_FINANCIEROS:
    print(f"[Prompt]  {p}")
    print(f"[Antes]   {gen_fin_antes[p]}")
    print(f"[Después] {gen_fin_despues[p]}\n")

### 15b. Extra: ¿el CPT financiero cambió cómo el modelo predice los tweets?

La reflexión del paso 9 habla de *olvido catastrófico*. Aquí lo medimos en vez de suponerlo: tomamos el modelo que acaba de adaptarse al dominio financiero y calculamos su perplexity sobre el holdout de **tweets** (el de la Parte A). El punto de comparación es el modelo base sobre esos mismos tweets (`ppl_antes`, paso 5), porque la Parte B arrancó desde el modelo base.

- Si la perplexity en tweets **sube**, adaptarse a finanzas le costó capacidad en el otro dominio.
- Si **baja**, el CPT financiero también le enseñó algo transferible (por ejemplo, español en general).

In [ ]:
# Modelo adaptado a FINANZAS, evaluado sobre el holdout de TWEETS (eval_loader, Parte A)
ppl_tweets_tras_cpt_fin = calcular_perplexity(model, eval_loader)

print(f"Perplexity en tweets, modelo base:               {ppl_antes:.2f}")
print(f"Perplexity en tweets, tras CPT con tweets:       {ppl_despues:.2f}")
print(f"Perplexity en tweets, tras CPT financiero:       {ppl_tweets_tras_cpt_fin:.2f}")
cambio_cruzado = 100 * (ppl_tweets_tras_cpt_fin - ppl_antes) / ppl_antes
print(f"Cambio cruzado (finanzas -> tweets): {cambio_cruzado:+.1f}%  (positivo = empeoró en tweets)")

### 15c. Tabla resumen de las dos adaptaciones

Una sola tabla con todos los números del lab, para sostener la reflexión final.

In [ ]:
import pandas as pd

resumen = pd.DataFrame(
    [
        {
            "dominio": "tweets (Parte A)",
            "chunks de entrenamiento": chunks_train.shape[0],
            "chunks de holdout": chunks_eval.shape[0],
            "epochs": EPOCHS,
            "lr": LR,
            "perplexity antes": round(ppl_antes, 2),
            "perplexity después": round(ppl_despues, 2),
            "reducción %": round(100 * (ppl_antes - ppl_despues) / ppl_antes, 1),
        },
        {
            "dominio": "financiero (Parte B)",
            "chunks de entrenamiento": chunks_fin.shape[0],
            "chunks de holdout": chunks_eval_fin.shape[0],
            "epochs": EPOCHS_FIN,
            "lr": LR_FIN,
            "perplexity antes": round(ppl_fin_antes, 2),
            "perplexity después": round(ppl_fin_despues, 2),
            "reducción %": round(100 * (ppl_fin_antes - ppl_fin_despues) / ppl_fin_antes, 1),
        },
    ]
).set_index("dominio")

display(resumen)
print(f"Cruzado: perplexity en tweets del modelo financiero = {ppl_tweets_tras_cpt_fin:.2f} "
      f"(base: {ppl_antes:.2f}, cambio {cambio_cruzado:+.1f}%)")

### 16. Reflexión final

```
# === PAUSA: DISCUSIÓN ===
```

Compara las dos adaptaciones:

- **Tweets vs. financiero:** ¿en cuál bajó más la perplexity? ¿Por qué? Piensa en el tamaño del corpus, qué tan "lejos" está cada dominio del preentrenamiento original del modelo, y qué tan homogéneo es cada texto.
- **Estilo generado:** ¿el texto financiero generado suena más formal que los tweets? El CPT no solo cambia vocabulario, también el **registro**.
- **Partir del modelo base:** como en esta parte recargaste el modelo base, tu comparación tweets-vs-financiero es **justa**: ambos CPT arrancaron del mismo punto. ¿Por qué habría sido engañoso entrenar el financiero encima del modelo ya adaptado a tweets?
- **En producción:** si tu empresa necesita un modelo experto en lenguaje financiero, ¿harías CPT como aquí, o combinarías el corpus financiero con datos generales para no perder capacidades? ¿Cuándo vale la pena CPT frente a *prompting* o RAG?
